# 03. 성적서에서 뭐가 읽히나

성적서 PDF 에는 **글자가 들어 있어 OCR 없이 읽힙니다** (정확도 100%).
이 노트북으로 파일 하나를 넣어 무엇이 뽑히는지 바로 확인할 수 있습니다.

읽히지 않는 항목이 있으면 `core/documents.py` 의 파서를 손보면 됩니다.

In [ ]:
# 이 셀을 먼저 실행하세요. 어디서 열어도 프로젝트를 찾습니다.
import sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent                      # notebooks/ 에서 열었을 때
assert (ROOT / "main.py").exists(), f"프로젝트를 찾지 못했습니다: {pathlib.Path.cwd()}"
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display

def 표(머리, 행들):
    """리스트를 표로 보여 준다."""
    if not 행들:
        display(Markdown("_내용 없음_"))
        return
    md = "| " + " | ".join(str(h) for h in 머리) + " |\n"
    md += "|" + "|".join("---" for _ in 머리) + "|\n"
    for r in 행들:
        md += "| " + " | ".join("" if c is None else str(c) for c in r) + " |\n"
    display(Markdown(md))

from core.config import load_config

# 설정은 현재 폴더 -> 프로젝트 폴더 순으로 찾고, 없으면 예시에서 만들어 준다
후보 = [pathlib.Path.cwd() / "config.yaml", ROOT / "config.yaml"]
cfg = load_config(next((p for p in 후보 if p.exists()), None))
print("프로젝트:", ROOT)
print("설정 파일:", cfg.source)

## 1. 읽을 파일 고르기

PDF 경로를 넣으세요. 탐색기에서 파일을 끌어다 놓으면 경로가 붙습니다.

In [ ]:
PDF = r""      # 예: r"E:\품질_전체\N. 의뢰시험\02. 의뢰시험 성적서\IS-2026-156157-00.pdf"

import pathlib
if not PDF:
    # 비워 두면 서류투입\성적서_* 폴더에서 아무거나 하나 집는다
    from core.paths import as_path
    뿌리 = as_path(cfg.path("intake_root"))
    후보 = [p for d in 뿌리.glob("성적서*") for p in d.rglob("*.pdf")] if 뿌리.exists() else []
    PDF = str(후보[0]) if 후보 else ""
print("읽을 파일:", PDF or "(없음 — 경로를 직접 넣으세요)")

## 2. 글자 뽑기

`pdf-text` 면 텍스트 레이어에서 읽은 것이고, `ocr:...` 면 스캔본이라
OCR 을 거친 것입니다. OCR 로 읽은 값은 꼭 눈으로 확인하세요.

In [ ]:
from core.extract import extract

got = extract(PDF) if PDF else None
if got:
    print("읽은 방식:", got)
    for w in got.warnings:
        print(" !", w)
    print("\n--- 앞부분 ---")
    print(got.text[:600])

## 3. 항목 뽑기

In [ ]:
from core.documents import parse_file

if PDF:
    doc, _ = parse_file(PDF, vendors=cfg.vendors)
    print("문서 종류:", doc.종류, f"(확신도 {doc.확신도:.2f})")
    표(["항목", "값"],
       [(k, v) for k, v in doc.항목.items() if v not in (None, "", [])])
    if doc.경고:
        print("확인 필요:")
        for w in doc.경고:
            print(" !", w)

### 읽히는 항목

| 문서 | 뽑는 것 |
|---|---|
| 아이텍 MT 성적서 | 성적서번호 · 접수일자 · 동 · 말뚝번호 · 결과 · 참관자 |
| 아이텍 재하 성적서 | 성적서번호 · 시험일자 · 동 · 말뚝번호 · **허용지지력** · 설계지지력 |
| PHC 생산 시험성적서 | 규격 · 로트번호 · 검사일자 · 바깥지름 · 두께 · 길이 |
| 자재 송장 | 업체 · 일자 · 규격 · 수량 |

재하 성적서의 **허용지지력**이 `O-01,04` 실시대장의 '시험 결과' 열에 들어가는 값입니다.

## 4. 여러 개 한꺼번에 확인

폴더를 통째로 훑어 무엇이 읽히고 무엇이 안 읽히는지 봅니다.

In [ ]:
폴더 = r""      # 비워 두면 서류투입\성적서_* 전체

from core.paths import as_path

if 폴더:
    파일들 = sorted(pathlib.Path(폴더).rglob("*.pdf"))
else:
    뿌리 = as_path(cfg.path("intake_root"))
    파일들 = sorted(p for d in 뿌리.glob("성적서*") for p in d.rglob("*.pdf")) if 뿌리.exists() else []

행 = []
for f in 파일들[:40]:
    doc, got = parse_file(f, vendors=cfg.vendors)
    행.append((f.name, doc.종류, f"{doc.확신도:.2f}", got.방식 or "실패",
               len([w for w in doc.경고 if not w.startswith("송장은")])))
표(["파일", "종류", "확신도", "읽은 방식", "경고"], 행)
print(len(파일들), "개 중", len(행), "개 확인")

## 5. 대장에 넣기

재하·MT 성적서는 `O-01,04` 실시대장에 바로 기록할 수 있습니다.
**Excel 이 필요하고, 값을 눈으로 확인한 뒤에만** 넣으세요.

In [ ]:
넣기 = False       # <- True 로 바꾸면 실제로 대장에 씁니다

from core.context import PreviewContext
from tasks.outsourced_report import OutsourcedReportTask, ReportEntry

if PDF:
    doc, _ = parse_file(PDF, vendors=cfg.vendors)
    entry = ReportEntry(doc=doc, 확인함=넣기)
    task = OutsourcedReportTask(cfg)
    문제 = task.validate_before(entry, PreviewContext(cfg))
    if 문제:
        print("먼저 확인할 것:")
        for p in 문제:
            print(" -", p)
    if 넣기 and not 문제:
        결과 = task.run(entry)
        print(결과.메시지)
    elif not 넣기:
        print("\n(넣기 = False 라 실제로는 쓰지 않았습니다)")